In [1]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "hybrid-search-rag"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,          # size of all-MiniLM-L6-v2 embeddings
        metric="dotproduct",    # required for hybrid (dense + sparse)
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

print("Index ready:", index_name)

Index ready: hybrid-search-rag


In [4]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever
from documents import chunks      # instead of: from documents import documents
       

load_dotenv()

# Connect to the index
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("hybrid-search-rag")

# Dense encoder (meaning)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


bm25_encoder = BM25Encoder().default()
bm25_encoder.fit(chunks)                 # train BM25 on the chunks
bm25_encoder.dump("bm25_values.json")
           # upload the chunks to Pinecone

100%|██████████| 15/15 [00:00<00:00, 1931.02it/s]


In [28]:
# retriever = PineconeHybridSearchRetriever(
#     embeddings=embeddings,
#     sparse_encoder=bm25_encoder,
#     index=index,
#     alpha=0.5,      # 0 = keyword only, 1 = semantic only
#     top_k=3,
# )

In [7]:
# Build the vectors manually (dense + sparse)
vectors = []
for i, text in enumerate(chunks):
    dense = embeddings.embed_query(text)             # HuggingFace dense vector
    sparse = bm25_encoder.encode_documents(text)     # BM25 sparse vector

    vectors.append({
        "id": f"chunk-{i}",
        "values": dense,
        "sparse_values": sparse,
        "metadata": {"context": text},               # must be named "context"
    })

# Upload with keyword argument
index.upsert(vectors=vectors)

print("Uploaded", len(vectors), "chunks")

Uploaded 15 chunks


In [15]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# 1. Connect to Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("hybrid-search-rag")

# 2. Load encoders (the saved BM25 file, not retrained)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
bm25_encoder = BM25Encoder().load("bm25_values.json")

# 3. Hybrid retriever
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    alpha=0.5,
    top_k=3,
)

# 4. Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1182.79it/s]


In [17]:
def get_text(content):
    """Return plain text from a Gemini response (string or list of blocks)."""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

In [21]:
def ask(question, show_sources=False):
    docs = retriever.invoke(question)
    context = "\n".join(d.page_content for d in docs)
    prompt = f"Answer using only this context:\n{context}\n\nQuestion: {question}\nAnswer:"
    answer = get_text(llm.invoke(prompt).content)

    if show_sources:
        print("\nSources:")
        for d in docs:
            print(f" - {d.page_content[:80]}  (score: {d.metadata.get('score'):.2f})")
    return answer

In [20]:
ask("What is BM25?")

'BM25 is a keyword search algorithm that gives a higher score to documents that contain the exact words from the query.'

In [23]:
ask("Why is combining two search methods better?" ,show_sources=True)


Sources:
 - Hybrid search combines keyword search and semantic search. It gives better resul  (score: 0.43)
 - BM25 is a keyword search algorithm. It gives a higher score to documents that co  (score: 0.21)
 - RAG means Retrieval Augmented Generation. First the system finds relevant docume  (score: 0.14)


'Because it finds both exact words and similar meanings.'

In [24]:
ask("How much water should I drink daily?")

'Based on the provided context, most adults should drink around two to three liters of water per day.'

In [25]:
ask("Tell me about Ahmedabad")

'Ahmedabad is a city located in Gujarat, India. It is known for its textile industry, the Sabarmati Riverfront, and its old walled city, which is a UNESCO World Heritage site.'

In [27]:
ask("who is Salman Khan")

'Based on the provided context, there is no information about who Salman Khan is.'